# Baseline Prediction Visualization

Load a trained baseline SwinUnet checkpoint, run predictions on the wildfire test split, and render compact sample sheets that show:

- all input feature groups
- landcover collapsed from 17 one-hot channels into one categorical tile
- previous-fire inputs
- ground-truth fire mask
- predicted fire probability
- predicted fire mask
- agreement / error map

The notebook saves each sample sheet to `results/prediction_visuals/`.


In [ ]:
from pathlib import Path

# Dataset / model config
HDF5_DIR = Path(r"G:\My Drive\wildfire_dataset\hdf5")
FOLD_ID = 0
N_LEADING_OBSERVATIONS = 1
USE_FACTORED_EMBED = False
CROP_SIDE_LENGTH = 128
BATCH_SIZE = 1
NUM_WORKERS = 0
SEED = 42

# Checkpoint location copied into this repo
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config.py").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing config.py")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RESULTS_BASE = REPO_ROOT / "results" / "wildfire_runs"
if not RESULTS_BASE.exists():
    RESULTS_BASE = REPO_ROOT / "results" / "wildfire_runs_baseline"
MODEL_PATH = RESULTS_BASE / f"fold{FOLD_ID}" / "best_model.pth"

# Visualization controls
SAMPLE_INDICES = None          # e.g. [5, 42, 81]; None = auto-pick interesting samples
NUM_SAMPLES = 4
SCAN_LIMIT = 256              # when auto-picking, scan first N test samples and keep the most fire-heavy
VIS_TIMESTEP = -1             # used when N_LEADING_OBSERVATIONS > 1
PRED_THRESHOLD = 0.5
SAVE_DIR = REPO_ROOT / "results" / "prediction_visuals"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{REPO_ROOT=}")
print(f"{MODEL_PATH=}")
print(f"{SAVE_DIR=}")

In [ ]:
import sys
import random
import types
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.backends.cudnn as cudnn
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import get_config
from datasets.wildfire import WildfireDataset, N_FEATURES_PER_TIMESTEP, get_year_split
from datasets.wildfire.FireSpreadDataset import FireSpreadDataset
from networks.vision_transformer import SwinUnet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

if torch.cuda.is_available():
    cudnn.benchmark = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

In [ ]:
# Feature metadata

FEATURE_NAMES = {
    0: "VIIRS M11",
    1: "VIIRS I2",
    2: "VIIRS I1",
    3: "NDVI",
    4: "EVI2",
    5: "Precip",
    6: "Wind speed",
    7: "Wind dir",
    8: "Min temp",
    9: "Max temp",
    10: "ERC",
    11: "Humidity",
    12: "Slope",
    13: "Aspect",
    14: "Elevation",
    15: "PDSI",
    33: "Fcst precip",
    34: "Fcst wind",
    35: "Fcst dir",
    36: "Fcst temp",
    37: "Fcst humid",
    38: "Prev fire age",
    39: "Prev fire mask",
}

LANDCOVER_LABELS = [
    "ENF", "EBF", "DNF", "DBF", "Mixed", "C Shrub", "O Shrub", "Wood Sav",
    "Savanna", "Grass", "Wetland", "Crop", "Urban", "Crop/Nat", "Snow/Ice",
    "Barren", "Water",
]

LANDCOVER_LONG = [
    "Evergreen Needleleaf Forest", "Evergreen Broadleaf Forest", "Deciduous Needleleaf Forest",
    "Deciduous Broadleaf Forest", "Mixed Forest", "Closed Shrublands", "Open Shrublands",
    "Woody Savannas", "Savannas", "Grasslands", "Permanent Wetlands", "Croplands",
    "Urban/Built-up", "Cropland/Natural Mosaic", "Permanent Snow/Ice", "Barren", "Water Bodies",
]

LANDCOVER_COLORS = [
    "#1b9e77", "#66a61e", "#7570b3", "#e7298a", "#a6761d", "#d95f02", "#e6ab02",
    "#a6cee3", "#1f78b4", "#b2df8a", "#33a02c", "#fb9a99", "#e31a1c", "#fdbf6f",
    "#cab2d6", "#6a3d9a", "#b15928",
]

LANDCOVER_CMAP = ListedColormap(LANDCOVER_COLORS)
MASK_CMAP = ListedColormap(["#d9d9d9", "#ef3b2c"])
AGREEMENT_CMAP = ListedColormap(["#d9d9d9", "#4daf4a", "#ff7f00", "#e41a1c"])
AGREEMENT_LABELS = {
    0: "TN",
    1: "TP",
    2: "FP",
    3: "FN",
}

FEATURE_LAYOUT = [
    [0, 1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10, 11],
    [12, 13, 14, 15, "landcover", 38],
    [33, 34, 35, 36, 37, 39],
    ["truth", "prob", "pred", "agreement", None, None],
]


def robust_normalize(arr):
    arr = np.asarray(arr, dtype=np.float32)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros_like(arr, dtype=np.float32)
    vals = arr[finite]
    lo = np.percentile(vals, 1)
    hi = np.percentile(vals, 99)
    if hi - lo < 1e-8:
        hi = vals.max()
        lo = vals.min()
        if hi - lo < 1e-8:
            return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def prepare_sample_features(x):
    if USE_FACTORED_EMBED:
        x_t = x[VIS_TIMESTEP].detach().cpu().numpy()
    else:
        x_t = x.reshape(N_LEADING_OBSERVATIONS, N_FEATURES_PER_TIMESTEP, x.shape[-2], x.shape[-1])[VIS_TIMESTEP]
        x_t = x_t.detach().cpu().numpy()
    return x_t


def make_landcover_tile(x_t):
    landcover_one_hot = x_t[16:33]
    return np.argmax(landcover_one_hot, axis=0)


def make_agreement_tile(y_true, y_pred):
    # 0=TN, 1=TP, 2=FP, 3=FN
    agreement = np.zeros_like(y_true, dtype=np.int64)
    agreement[(y_true == 1) & (y_pred == 1)] = 1
    agreement[(y_true == 0) & (y_pred == 1)] = 2
    agreement[(y_true == 1) & (y_pred == 0)] = 3
    return agreement

In [ ]:
def build_config():
    args = types.SimpleNamespace(
        cfg=str(REPO_ROOT / "configs" / "swin_tiny_patch4_window4_128_wildfire.yaml"),
        opts=[
            "MODEL.SWIN.IN_CHANS", str(N_FEATURES_PER_TIMESTEP),
            "MODEL.SWIN.N_TIMESTEPS", str(N_LEADING_OBSERVATIONS),
            "MODEL.PRETRAIN_CKPT", "None",
        ],
        batch_size=None,
        zip=False,
        cache_mode="part",
        resume=None,
        accumulation_steps=None,
        use_checkpoint=False,
        amp_opt_level="O1",
        tag=None,
        eval=False,
        throughput=False,
    )
    return get_config(args)


def load_model(model_path: Path):
    config = build_config()
    net = SwinUnet(
        config,
        img_size=config.DATA.IMG_SIZE,
        num_classes=2,
        use_factored_embed=USE_FACTORED_EMBED,
    ).to(DEVICE)

    state = torch.load(model_path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state" in state:
        state = state["model_state"]
    if next(iter(state)).startswith("module."):
        state = {k.replace("module.", ""): v for k, v in state.items()}
    net.load_state_dict(state)
    net.eval()
    return net


def build_test_dataset():
    train_years, _, test_years = get_year_split(FOLD_ID)
    ds = WildfireDataset(
        data_dir=str(HDF5_DIR),
        included_fire_years=test_years,
        is_train=False,
        stats_years=train_years,
        n_leading_observations=N_LEADING_OBSERVATIONS,
        n_leading_observations_test_adjustment=N_LEADING_OBSERVATIONS,
        crop_side_length=CROP_SIDE_LENGTH,
        load_from_hdf5=True,
        use_factored_embed=USE_FACTORED_EMBED,
    )
    return ds, train_years, test_years


model = load_model(MODEL_PATH)
dataset, train_years, test_years = build_test_dataset()
channel_map = FireSpreadDataset.map_channel_index_to_features()

print(f"Loaded model from: {MODEL_PATH}")
print(f"Test years: {test_years}")
print(f"Dataset size: {len(dataset)}")

In [ ]:
def choose_sample_indices(dataset, sample_indices=None, num_samples=4, scan_limit=256):
    if sample_indices is not None:
        return sample_indices

    limit = min(len(dataset), scan_limit)
    scored = []
    for idx in range(limit):
        _, y = dataset[idx]
        fire_frac = float(y.float().mean().item())
        scored.append((fire_frac, idx))
    scored.sort(reverse=True)
    picked = [idx for _, idx in scored[:num_samples]]
    return picked


sample_indices = choose_sample_indices(
    dataset,
    sample_indices=SAMPLE_INDICES,
    num_samples=NUM_SAMPLES,
    scan_limit=SCAN_LIMIT,
)
print("Selected sample indices:", sample_indices)

In [ ]:
def plot_case_sheet(x, y_true, prob_fire, pred_mask, case_title, save_path=None):
    x_t = prepare_sample_features(x)
    y_true = y_true.detach().cpu().numpy().astype(np.int64)
    prob_fire = prob_fire.detach().cpu().numpy()
    pred_mask = pred_mask.detach().cpu().numpy().astype(np.int64)
    landcover = make_landcover_tile(x_t)
    agreement = make_agreement_tile(y_true, pred_mask)

    fig, axes = plt.subplots(
        nrows=len(FEATURE_LAYOUT),
        ncols=len(FEATURE_LAYOUT[0]),
        figsize=(18, 14),
        constrained_layout=False,
    )
    fig.subplots_adjust(left=0.04, right=0.80, top=0.92, bottom=0.05, wspace=0.08, hspace=0.28)
    fig.suptitle(case_title, fontsize=16, fontweight="bold")

    for r, row in enumerate(FEATURE_LAYOUT):
        for c, item in enumerate(row):
            ax = axes[r, c]
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

            if item is None:
                ax.axis("off")
                continue

            if item == "landcover":
                ax.imshow(landcover, cmap=LANDCOVER_CMAP, vmin=0, vmax=16, interpolation="nearest")
                ax.set_title("Landcover", fontsize=10)
            elif item == "truth":
                ax.imshow(y_true, cmap=MASK_CMAP, vmin=0, vmax=1, interpolation="nearest")
                ax.set_title("Fire mask", fontsize=10)
            elif item == "pred":
                ax.imshow(pred_mask, cmap=MASK_CMAP, vmin=0, vmax=1, interpolation="nearest")
                ax.set_title("Pred mask", fontsize=10)
            elif item == "prob":
                ax.imshow(prob_fire, cmap="magma", vmin=0.0, vmax=1.0, interpolation="nearest")
                ax.set_title("Pred prob", fontsize=10)
            elif item == "agreement":
                ax.imshow(agreement, cmap=AGREEMENT_CMAP, vmin=0, vmax=3, interpolation="nearest")
                ax.set_title("Agreement", fontsize=10)
            else:
                arr = x_t[item]
                cmap = "twilight" if item in {7, 13, 35} else "viridis"
                ax.imshow(robust_normalize(arr), cmap=cmap, interpolation="nearest")
                ax.set_title(FEATURE_NAMES.get(item, channel_map[item]), fontsize=10)

    landcover_handles = [
        Patch(facecolor=color, edgecolor="none", label=label)
        for color, label in zip(LANDCOVER_COLORS, LANDCOVER_LABELS)
    ]
    mask_handles = [
        Patch(facecolor="#ef3b2c", edgecolor="none", label="Fire / predicted fire"),
        Patch(facecolor="#d9d9d9", edgecolor="none", label="Background"),
        Patch(facecolor="#4daf4a", edgecolor="none", label="TP"),
        Patch(facecolor="#ff7f00", edgecolor="none", label="FP"),
        Patch(facecolor="#e41a1c", edgecolor="none", label="FN"),
    ]

    fig.legend(
        handles=landcover_handles,
        loc="upper left",
        bbox_to_anchor=(0.815, 0.88),
        fontsize=8,
        title="Landcover",
        title_fontsize=10,
        ncols=2,
        frameon=False,
    )
    fig.legend(
        handles=mask_handles,
        loc="upper left",
        bbox_to_anchor=(0.815, 0.42),
        fontsize=8,
        title="Outputs",
        title_fontsize=10,
        frameon=False,
    )

    if save_path is not None:
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


def run_case(index):
    x, y_true = dataset[index]
    x_in = x.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x_in)
        prob_fire = torch.softmax(logits, dim=1)[0, 1]
    pred_mask = (prob_fire >= PRED_THRESHOLD).long()

    fire_frac = float(y_true.float().mean().item())
    pred_frac = float(pred_mask.float().mean().item())
    save_path = SAVE_DIR / f"fold{FOLD_ID}_sample{index}.png"
    title = (
        f"Fold {FOLD_ID} | Test sample {index} | "
        f"true fire={fire_frac:.3f}, pred fire={pred_frac:.3f}, "
        f"threshold={PRED_THRESHOLD:.2f}"
    )
    plot_case_sheet(x, y_true, prob_fire, pred_mask, title, save_path=save_path)


for idx in sample_indices:
    run_case(idx)